In [1]:
from google.colab import files
uploaded = files.upload()

Saving archive (2).zip to archive (2).zip


In [2]:
import zipfile
import os
zip_file = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall()
print("Files after extraction:")
print(os.listdir())

Files after extraction:
['.config', 'archive (2).zip', 'True.csv', 'Fake.csv', 'sample_data']


In [3]:
import os
for root, dirs, files in os.walk("."):
    for file in files:
        print(os.path.join(root, file))

./True.csv
./Fake.csv
./archive.zip
./.config/config_sentinel
./.config/.last_update_check.json
./.config/default_configs.db
./.config/.last_opt_in_prompt.yaml
./.config/active_config
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/.last_survey_prompt.yaml
./.config/gce
./.config/configurations/config_default
./.config/logs/2026.02.06/14.31.45.734270.log
./.config/logs/2026.02.06/14.31.35.535753.log
./.config/logs/2026.02.06/14.30.32.592228.log
./.config/logs/2026.02.06/14.31.28.771044.log
./.config/logs/2026.02.06/14.31.19.332851.log
./.config/logs/2026.02.06/14.31.44.938153.log
./sample_data/anscombe.json
./sample_data/README.md
./sample_data/california_housing_test.csv
./sample_data/mnist_test.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_train.csv


In [4]:
import pandas as pd
fake = pd.read_csv("Fake.csv").head(1000)
real = pd.read_csv("True.csv").head(1000)

In [5]:
!pip install transformers datasets torch scikit-learn

In [6]:
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments
fake["label"] = 0
real["label"] = 1
data = pd.concat([fake, real])
data = data.sample(frac=1).reset_index(drop=True)
data = data[["text", "label"]]
train_texts, val_texts, train_labels, val_labels = train_test_split(
    data["text"].tolist(),
    data["label"].tolist(),
    test_size=0.2
)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = NewsDataset(train_encodings, train_labels)
val_dataset = NewsDataset(val_encodings, val_labels)
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)
trainer.train()
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.0001488003763370216,
 'eval_runtime': 5.9378,
 'eval_samples_per_second': 67.366,
 'eval_steps_per_second': 8.421,
 'epoch': 2.0}

In [13]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {key: val.to(model.device) for key, val in inputs.items()}
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()
    return "Real News" if pred == 1 else "Fake News"

print(predict("Senate leader McConnell sees a more collegial 2018"))

Real News


In [16]:
model.save_pretrained("model")
tokenizer.save_pretrained("model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('model/tokenizer_config.json', 'model/tokenizer.json')

In [17]:
!zip -r model.zip model

  adding: model/ (stored 0%)
  adding: model/config.json (deflated 49%)
  adding: model/model.safetensors (deflated 8%)
  adding: model/tokenizer.json (deflated 71%)
  adding: model/tokenizer_config.json (deflated 42%)


In [18]:
from google.colab import files
files.download("model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>